In [1]:
from decouple import config
from google import genai
from google.genai import types
from IPython.display import Markdown, display

In [2]:
GOOGLE_API_KEY = config("GOOGLE_API_KEY_NEW")
client = genai.Client(api_key=GOOGLE_API_KEY)
GOOGLE_GEMINI_MODEL1 = config("GOOGLE_GEMINI_MODEL1")
GOOGLE_GEMINI_MODEL2 = config("GOOGLE_GEMINI_MODEL2")

In [3]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '11',
  'version': '10.0.26200',
  'kernel': '11',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'mingw32'},
 'package_managers': ['winget'],
 'cpu': {'brand': '12th Gen Intel(R) Core(TM) i5-12450HX',
  'cores_logical': 12,
  'cores_physical': 8,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'gcc.exe (MinGW.org GCC-6.3.0-1) 6.3.0',
   'g++': 'g++.exe (MinGW.org GCC-6.3.0-1) 6.3.0',
   'clang': '',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

In [4]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""
# response = client.models.generate_content(
#     model=GOOGLE_GEMINI_MODEL2,
#     contents=message
# )
# display(Markdown(response.text))

In [5]:
source_file = "main.cpp"
executable_file = "main.exe" # On Windows, executables typically have a .exe extension

compile_command = [
    "g++",
    source_file,
    "-o", executable_file,
    "-O3",
    "-march=native",
    "-mtune=native",
    "-s",
    "-std=c++17"
]
run_command = [f".\\{executable_file}"] # Use f-string for clarity and Windows path


In [6]:
system_prompt = """
Your task is to convert python code into high perYour task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""
def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [7]:
def write_cpp_output(cpp : str):
    with open("main.cpp", "w") as f:
        f.write(cpp)

In [8]:
def  port_python_to_cpp(python_code: str):
    response = client.models.generate_content(
        model=GOOGLE_GEMINI_MODEL1,
        contents = user_prompt_for(python=python_code),
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
        )
    )
    output = response.text
    output = output.replace("```cpp", "").replace("```", "").strip()
    write_cpp_output(output)
    return output

In [9]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [10]:
import io
import sys


def run_python_code(python_code: str):
    globals = {"__builtins__": __builtins__}
    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer
    try:
        exec(python_code, globals)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {str(e)}"
    finally:
        sys.stdout = old_stdout
    return output

In [ ]:
run_python_code("print('sum:', end='');print(2+3)")

'sum:5\n'

In [14]:
run_python_code(python_code=pi)

'Result: 3.141592656089\nExecution Time: 36.377441 seconds\n'

In [16]:
cpp = port_python_to_cpp(python_code=pi)
cpp

'#include <iostream>\n#include <chrono>\n#include <iomanip>\n#include <immintrin.h>\n\n// This version is single-threaded but uses AVX2 intrinsics for vectorization,\n// combined with loop unrolling to maximize instruction-level parallelism.\n// Since the provided compile command does not include a flag for parallelization\n// (like -fopenmp), this single-threaded implementation aims for maximum performance.\ndouble calculate(long long iterations, double param1, double param2) {\n    // Vector constants that will be loaded into registers\n    const __m256d v_p2 = _mm256_set1_pd(param2);\n    const __m256d v_one = _mm256_set1_pd(1.0);\n    \n    // The loop is optimized by directly incrementing the product `i * param1`\n    // instead of incrementing `i` and then multiplying inside the loop.\n    // Step for a single vector of 4 doubles\n    const __m256d v_step1 = _mm256_set1_pd(4.0 * param1);\n    // Step for the unrolled loop (processing 2 vectors, i.e., 8 doubles)\n    const __m256d

In [17]:
import subprocess
def compile_and_run_cpp(compile_cmd = compile_command, run_command = run_command):
    try:
        subprocess.run(compile_cmd, check=True, text=True, capture_output=True)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    except subprocess.CalledProcessError as e:
        print(f"An error occurred:\n{e.stderr}")


In [19]:
compile_and_run_cpp()

Result: 3.141592656315
Execution Time: 0.200219 seconds

Result: 3.141592656315
Execution Time: 0.202035 seconds

Result: 3.141592656315
Execution Time: 0.195623 seconds

